# Face Recognition — Two-Model Pipeline

**Projet** : Système de Reconnaissance Faciale — PFE Sonatel Academy  
**Auteur** : Ibrahima Gabar Diop  

## Architecture



Ce notebook entraîne **uniquement le Modèle 2** (classificateur).  
Le Modèle 1 est le poids pré-entraîné  téléchargé automatiquement.


In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ── Imports & constants ────────────────────────────────────────────────────
import os, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from ultralytics import YOLO

# Paths Kaggle
LFW_DIR      = Path("/kaggle/input/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled")
CSV_ALLNAMES = Path("/kaggle/input/lfw-dataset/lfw_allnames.csv")
OUTPUT_DIR   = Path("/kaggle/working")
DATA_DIR     = OUTPUT_DIR / "lfw_cls"         # structure classify
MODEL_DIR    = OUTPUT_DIR / "runs"

# Hyperparamètres
N_CLASSES    = 20
MIN_IMAGES   = 30
TRAIN_RATIO  = 0.70
VAL_RATIO    = 0.15
# test = 1 - train - val
BASE_MODEL   = "yolo11m-cls.pt"               # classificateur pré-entraîné ImageNet
RUN_NAME     = "face_classifier_lfw20"
EPOCHS       = 50
IMG_SIZE     = 224
BATCH        = 64
SEED         = 42
random.seed(SEED)
np.random.seed(SEED)


## 1. Exploration du dataset LFW


In [ ]:
df_all = pd.read_csv(CSV_ALLNAMES)
print(df_all.head())
print(f"Total personnes : {len(df_all)}")
print(f"Total images    : {df_all['images'].sum()}")

eligible = df_all[df_all["images"] >= MIN_IMAGES].sort_values("images", ascending=False)
print(f"
Personnes avec >= {MIN_IMAGES} images : {len(eligible)}")


In [ ]:
top20 = eligible.head(N_CLASSES)
print(top20[["name","images"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(top20["name"][::-1], top20["images"][::-1], color="steelblue")
ax.set_xlabel("Nombre d'images")
ax.set_title(f"LFW Top-{N_CLASSES} — distribution des images")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_balance.png", dpi=150)
plt.show()


## 2. Construction du dataset (format classification)

Structure attendue par  :



In [ ]:
# Créer la structure classify ───────────────────────────────────────────────
for split in ["train", "val", "test"]:
    for _, row in top20.iterrows():
        (DATA_DIR / split / row["name"]).mkdir(parents=True, exist_ok=True)

copied = {split: 0 for split in ["train", "val", "test"]}

for _, row in top20.iterrows():
    person = row["name"]
    imgs = sorted((LFW_DIR / person).glob("*.jpg"))
    random.shuffle(imgs)

    n = len(imgs)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)

    splits_map = (
        [("train", imgs[:n_train])]
        + [("val",   imgs[n_train:n_train+n_val])]
        + [("test",  imgs[n_train+n_val:])]
    )
    for split, files in splits_map:
        for f in files:
            shutil.copy(f, DATA_DIR / split / person / f.name)
            copied[split] += 1

print("Images copiées :", copied)
print(f"Total : {sum(copied.values())}")


In [ ]:
# Vérification visuelle — 4 exemples par classe ─────────────────────────────
from PIL import Image
classes = [p.name for p in (DATA_DIR / "train").iterdir() if p.is_dir()]
fig, axes = plt.subplots(len(classes), 4, figsize=(12, len(classes)*2))
for i, cls in enumerate(sorted(classes)):
    imgs = list((DATA_DIR / "train" / cls).glob("*.jpg"))[:4]
    for j, img_path in enumerate(imgs):
        ax = axes[i][j]
        ax.imshow(Image.open(img_path))
        ax.axis("off")
        if j == 0:
            ax.set_title(cls[:20], fontsize=7, loc="left")
plt.suptitle("Exemples d'images par classe", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_samples.png", dpi=120)
plt.show()


## 3. Entraînement — Classificateur YOLOv8m-cls


In [ ]:
model = YOLO(BASE_MODEL)
print(model.info())


In [ ]:
results = model.train(
    data    = str(DATA_DIR),
    task    = "classify",
    epochs  = EPOCHS,
    imgsz   = IMG_SIZE,
    batch   = BATCH,
    lr0     = 0.001,
    lrf     = 0.01,
    patience= 15,
    project = str(MODEL_DIR),
    name    = RUN_NAME,
    seed    = SEED,
    verbose = True,
)


In [ ]:
# Copier le meilleur modèle ─────────────────────────────────────────────────
best_pt = MODEL_DIR / RUN_NAME / "weights" / "best.pt"
dest    = OUTPUT_DIR / "face_classifier.pt"
shutil.copy(best_pt, dest)
print(f"Modèle sauvegardé : {dest} ({dest.stat().st_size/1e6:.1f} MB)")


## 4. Courbes d'entraînement


In [ ]:
csv_path = MODEL_DIR / RUN_NAME / "results.csv"
df_res = pd.read_csv(csv_path)
df_res.columns = df_res.columns.str.strip()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(df_res["epoch"], df_res["train/loss"], label="Train loss")
axes[0].plot(df_res["epoch"], df_res["val/loss"],   label="Val loss", linestyle="--")
axes[0].set_xlabel("Époque")
axes[0].set_title("Loss")
axes[0].legend()

# Accuracy
axes[1].plot(df_res["epoch"], df_res["metrics/accuracy_top1"], label="Top-1 Acc")
axes[1].plot(df_res["epoch"], df_res["metrics/accuracy_top5"], label="Top-5 Acc", linestyle="--")
axes[1].set_xlabel("Époque")
axes[1].set_title("Accuracy")
axes[1].legend()

plt.suptitle("Courbes d'entraînement — face_classifier", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=150)
plt.show()


## 5. Évaluation — Test set


In [ ]:
classifier = YOLO(str(dest))
test_results = classifier.val(
    data  = str(DATA_DIR),
    split = "test",
    imgsz = IMG_SIZE,
)
print("Top-1 accuracy (test):", test_results.top1)
print("Top-5 accuracy (test):", test_results.top5)


In [ ]:
# Matrice de confusion ───────────────────────────────────────────────────────
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true, y_pred = [], []
test_classes = sorted([p.name for p in (DATA_DIR / "test").iterdir() if p.is_dir()])

for cls in test_classes:
    for img_path in (DATA_DIR / "test" / cls).glob("*.jpg"):
        r = classifier.predict(str(img_path), verbose=False)[0]
        pred = r.names[r.probs.top1]
        y_true.append(cls)
        y_pred.append(pred)

cm = confusion_matrix(y_true, y_pred, labels=test_classes)
fig, ax = plt.subplots(figsize=(14, 12))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[c[:15] for c in test_classes])
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
plt.title("Matrice de confusion — Test set")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=150)
plt.show()


## 6. Export ONNX


In [ ]:
onnx_path = classifier.export(format="onnx", imgsz=IMG_SIZE)
print(f"ONNX exporté : {onnx_path}")


## 7. Résumé


In [ ]:
print("=" * 55)
print("  RÉSUMÉ FINAL")
print("=" * 55)
print(f"  Modèle         : YOLOv8m-cls")
print(f"  Classes        : {N_CLASSES}")
print(f"  Images train   : {copied[chr(116)+chr(114)+chr(97)+chr(105)+chr(110)]}")
print(f"  Images val     : {copied[chr(118)+chr(97)+chr(108)]}")
print(f"  Images test    : {copied[chr(116)+chr(101)+chr(115)+chr(116)]}")
print(f"  Top-1 (test)   : {test_results.top1:.4f}")
print(f"  Top-5 (test)   : {test_results.top5:.4f}")
print(f"  Modèle         : /kaggle/working/face_classifier.pt")
print(f"  ONNX           : /kaggle/working/face_classifier.onnx")
print("=" * 55)
print("Télécharger face_classifier.pt et le placer dans models/")
